# Audit: data construction and robustness of the commune density–coordination result

Preliminary results (`analysis_communes.ipynb`): little or no density–CENP association in 2002 once very small
communes are excluded, a positive one in 2022, stronger for larger electorates. This notebook audits the data
construction and re-estimates the relationship under alternative, pre-specified choices:

| § | check |
|---|---|
| 1 | failed commune scrapes |
| 2 | why observations leave `main_sample` (one reason per observation) |
| 3 | boundary-change rule: **strict** (`main_sample`) vs **broad** cross-sectional sample |
| 4 | which population measure enters the 2002 density |
| 5 | density validation, Paris / Lyon / Marseille |
| 6 | small electorates and extreme CENP values |
| 7–8 | electorate-size thresholds; vote-weighted robustness |
| 9 | density deciles |
| 10 | within-département (fixed-effects) association |
| 11 | pooled year × density interaction |
| 12 | cliff measures (secondary) |
| Summary | computed summary of everything above |

Rules: 2002 and 2022 are **two separate cross-sections**, not a matched panel; all thresholds and samples are reported
side by side, none is chosen for its result; CENP is the main measure; everything is descriptive.

Outputs (`data/processed/audit_corrected/`): `audit_cenp_density_by_threshold.csv`, `audit_scorecard.csv`,
`audit_sample_flags.csv`. Nothing is downloaded: pages are read from the HTML cache only.

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

from svgeo import communes as com
from svgeo.analysis import association_stats, fe_stats, show_plot, stats_by_threshold
from svgeo.config import (ARRONDISSEMENTS, AUDIT_DIR, COMMUNE_DENSITY, COMMUNE_FAILURES, COMMUNE_RESULTS,
                          EXPECTED_K, INSEE_COMMUNES, INSEE_COMMUNES_META, PLM, YEARS)
from svgeo.utils import norm_text, read_csv, report
from svgeo.web import commune_pages

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
commune_pages.offline = True  # cached pages only

ALPHA = 0.05
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

results = read_csv(COMMUNE_RESULTS)
merged = read_csv(COMMUNE_DENSITY)
main = merged[merged["main_sample"]].copy()
scrape_failures = pd.concat([read_csv(COMMUNE_FAILURES[y]) for y in YEARS], ignore_index=True)

insee_raw = pd.read_csv(INSEE_COMMUNES, sep=";", dtype={"CODGEO": str}, encoding="utf-8",
                        usecols=["CODGEO", "P22_POP", "P06_POP", "D99_POP", "SUPERF"])
insee = (insee_raw[~insee_raw["CODGEO"].isin(ARRONDISSEMENTS)]
         .rename(columns={"CODGEO": "commune_code", "SUPERF": "surface_km2"}))
# Commune names of the INSEE file (its metadata lists every CODGEO with its name)
insee_names = (pd.read_csv(INSEE_COMMUNES_META, sep=";", dtype=str, encoding="utf-8")
                 .query("COD_VAR == 'CODGEO'")[["COD_MOD", "LIB_MOD"]]
                 .rename(columns={"COD_MOD": "commune_code", "LIB_MOD": "insee_name"})
                 .drop_duplicates("commune_code"))

## 1. Failed commune scrapes

The failure files list the pages still failing **after the retries**. "Expected communes" = commune URLs found on the
département / alphabetical index pages: set `REBUILD_DISCOVERY_FROM_CACHE = True` to rebuild that list from the HTML
cache (a few minutes); otherwise "parsed + recorded commune failures" is used. `no communes` entries are territories
whose results are not published by commune. Two further checks: pages read correctly but reporting **0 expressed
votes**, and INSEE communes without any election page.

In [ ]:
REBUILD_DISCOVERY_FROM_CACHE = False

with pd.option_context("display.max_colwidth", 120):
    display(scrape_failures.sort_values(["year", "level", "department_code"]).reset_index(drop=True))

scrape_rows, not_parsed = [], {}
for year in YEARS:
    parsed = set(results.loc[results["year"] == year, "commune_code"])
    f = scrape_failures[scrape_failures["year"] == year]
    commune_fail = f[f["level"] == "commune"]
    if REBUILD_DISCOVERY_FROM_CACHE:
        urls = com.discover_all_communes(year, com.load_department_urls(year), verbose=False)[0]
        expected, source = set(urls["commune_code"]), "commune URLs rebuilt from the HTML cache"
        not_parsed[year] = urls[~urls["commune_code"].isin(parsed)]
    else:
        expected = parsed | set(commune_fail["commune_code"].dropna())
        source = "parsed + recorded commune failures"
    scrape_rows.append({
        "year": year,
        "expected_communes": len(expected),
        "scraped_communes": len(expected & parsed),
        "failed_communes": len(expected - parsed),
        "pct_recovered": round(100 * len(expected & parsed) / len(expected), 3),
        "recorded_commune_page_failures": len(commune_fail),
        "failed_departement_or_index_pages": int(f["level"].isin(["department", "index page"]).sum()),
        "territories_without_commune_pages": int((f["level"] == "no communes").sum()),
        "expected_from": source,
    })
scrape_summary = pd.DataFrame(scrape_rows).set_index("year")
display(scrape_summary)
for year, df in not_parsed.items():
    report(f"{year}: every discovered commune URL has parsed results", df.empty, df)

In [ ]:
# Pages read correctly but unusable: 0 expressed votes (every ballot blank/null — typically results annulled)
urls_by_commune = results.drop_duplicates(["year", "commune_code"])[["year", "commune_code", "source_url"]]
zero_expressed = (merged.loc[merged["expressed"] <= 0, ["year", "department_code", "commune_code", "commune_name",
                                                        "registered", "voters", "blank_null", "expressed"]]
                  .merge(urls_by_commune, on=["year", "commune_code"], how="left"))
print(f"Communes with 0 expressed votes: {len(zero_expressed)}")
with pd.option_context("display.max_colwidth", 120):
    display(zero_expressed)

for year in YEARS:
    missing = (insee[~insee["commune_code"].isin(results.loc[results["year"] == year, "commune_code"])]
               .merge(insee_names, on="commune_code", how="left"))
    print(f"\n{year}: INSEE communes without election results: {len(missing)} (codes created after {year}, ...)")
    with pd.option_context("display.max_rows", 100):
        display(missing[["commune_code", "insee_name", "D99_POP", "P06_POP", "P22_POP", "surface_km2"]])

**Cross-sections, not a panel.** A commune missing in one year (failed scrape, annulled results, no INSEE match) is
excluded from that year only; restricting both years to a common set of communes would discard valid information.

In [ ]:
codes_in_main = {year: set(main.loc[main["year"] == year, "commune_code"]) for year in YEARS}
print("main_sample communes: only in 2002:", len(codes_in_main[2002] - codes_in_main[2022]),
      "| only in 2022:", len(codes_in_main[2022] - codes_in_main[2002]),
      "| in both years:", len(codes_in_main[2002] & codes_in_main[2022]), "— all kept in their own year")

## 2. Why observations leave `main_sample`

`main_sample` = metropolitan **and** matched to INSEE **and** finite `log_density` **and** CENP available **and not**
`possible_boundary_change`. Each observation receives the **first** reason that applies, in the order below, so
reasons add up to the total. Unmatched codes are split using the 2022 results as a diagnostic: a 2002 code absent from
the 2022 results disappeared in a merger between 2002 and 2022; a code present in the 2022 results but not in the
INSEE file disappeared **after** the 2022 election (the INSEE geography is later than 2022 — see §3). The reasons are
checked to reproduce `main_sample` exactly.

In [ ]:
def name_key(name):
    # commune name normalised for comparison: accents, case, punctuation, St/Ste, article in parentheses
    s = str(name).replace("œ", "oe").replace("Œ", "OE").replace("æ", "ae").replace("Æ", "AE")
    m = re.fullmatch(r"(.*?)\s*\((l'|l’|le|la|les)\)\s*", s, flags=re.I)
    if m:
        s = f"{m.group(2)} {m.group(1)}"
    s = norm_text(s)
    s = re.sub(r"\bste\b", "sainte", s)
    s = re.sub(r"\bst\b", "saint", s)
    return s.replace(" ", "")


audit = merged.merge(insee_names, on="commune_code", how="left", validate="many_to_one")
audit["valid_density"] = (audit["insee_match"] & (audit["density_population"] > 0) & (audit["surface_km2"] > 0)
                          & np.isfinite(audit["log_density"]))
audit["ratio_low"] = audit["insee_match"] & (audit["registered_to_population"] < 0.4)
audit["ratio_high"] = audit["insee_match"] & (audit["registered_to_population"] > 1.2)
audit["name_mismatch"] = audit["insee_match"] & (audit["commune_name"].map(name_key) != audit["insee_name"].map(name_key))

unmatched_deps = set(map(tuple, audit.loc[audit["metropolitan"] & ~audit["insee_match"],
                                          ["year", "department_code"]].itertuples(index=False)))
audit["dep_has_unmatched"] = [(y, d) in unmatched_deps for y, d in zip(audit["year"], audit["department_code"])]
codes_2022 = set(results.loc[results["year"] == 2022, "commune_code"])

RETAINED = "retained (main_sample)"
EXCLUSION_RULES = [
    ("overseas", ~audit["metropolitan"]),
    ("0 expressed votes (all ballots blank/null)", audit["expressed"].fillna(0) <= 0),
    ("indices missing (other)", audit["CENP"].isna()),
    ("Paris/Lyon/Marseille arrondissement code", audit["commune_code"].isin(ARRONDISSEMENTS)),
    ("no INSEE match: 2002 code absent from 2022 results (merger 2002–2022)",
     ~audit["insee_match"] & (audit["year"] == 2002) & ~audit["commune_code"].isin(codes_2022)),
    ("no INSEE match: code in 2022 results, not in INSEE file (merger after 2022)", ~audit["insee_match"]),
    ("INSEE population missing", audit["population"].isna()),
    ("INSEE population = 0", audit["population"] <= 0),
    ("INSEE surface missing or <= 0", ~(audit["surface_km2"] > 0)),
    ("non-finite log density (other)", ~np.isfinite(audit["log_density"])),
    ("boundary flag: registered/population < 0.4", audit["registered_to_population"] < 0.4),
    ("boundary flag: registered/population > 1.2", audit["registered_to_population"] > 1.2),
]
audit["exclusion_reason"] = np.select([cond.to_numpy(dtype=bool) for _, cond in EXCLUSION_RULES],
                                      [label for label, _ in EXCLUSION_RULES], default=RETAINED)

mismatch = (audit["exclusion_reason"] == RETAINED) != audit["main_sample"]
report("exclusion reasons reproduce main_sample exactly", not mismatch.any(),
       audit.loc[mismatch, ["year", "commune_code", "commune_name", "exclusion_reason", "main_sample"]])

reason_order = [label for label, _ in EXCLUSION_RULES] + [RETAINED]
exclusion_counts = pd.crosstab(audit["exclusion_reason"], audit["year"]).reindex(reason_order, fill_value=0)
metro_totals = audit[audit["metropolitan"]].groupby("year").size()
exclusion_pct_metro = (exclusion_counts.drop(index="overseas") / metro_totals * 100).round(2)
print("Observations by primary exclusion reason:")
display(pd.concat({"n": exclusion_counts, "% of metropolitan": exclusion_pct_metro}, axis=1))

metro = audit[audit["metropolitan"]]
sample_counts = metro.groupby("year").agg(
    metropolitan_election_obs=("commune_code", "size"), with_CENP=("CENP", lambda s: int(s.notna().sum())),
    insee_matched=("insee_match", "sum"), valid_density=("valid_density", "sum"), main_sample=("main_sample", "sum"))
sample_counts["pct_retained"] = (100 * sample_counts["main_sample"] / sample_counts["metropolitan_election_obs"]).round(2)
display(sample_counts)

print("Non-exclusive flags among metropolitan observations:")
display(metro.assign(no_insee_match=~metro["insee_match"], zero_expressed=metro["expressed"].fillna(0) <= 0,
                     invalid_density=~metro["valid_density"])
        .groupby("year")[["zero_expressed", "no_insee_match", "invalid_density", "ratio_low", "ratio_high",
                          "possible_boundary_change", "name_mismatch"]].sum())

print("Are the excluded observations different? (median log density / expressed, mean CENP by reason)")
display(metro.groupby(["year", "exclusion_reason"]).agg(
    n=("commune_code", "size"), median_log_density=("log_density", "median"),
    median_expressed=("expressed", "median"), mean_CENP=("CENP", "mean")).round(3))

In [ ]:
EXAMPLE_COLS = ["year", "department_code", "commune_code", "commune_name", "insee_name", "registered", "expressed",
                "population", "surface_km2", "registered_to_population", "density", "CENP"]
for (year, reason), g in audit[audit["exclusion_reason"] != RETAINED].groupby(["year", "exclusion_reason"]):
    print(f"\n{year} — {reason}: {len(g)} observations (up to 10, largest electorates first)")
    display(g.sort_values("registered", ascending=False)[EXAMPLE_COLS].head(10))

## 3. Boundary changes: strict vs broad cross-sectional sample

The INSEE populations and surfaces are in one recent commune geography. The flag `possible_boundary_change`
excludes every matched commune with registered / population outside [0.4, 1.2], in both years. Problems:

* **> 1.2** is not a sign of a boundary change (a *larger* INSEE unit makes the ratio too **low**); it typically
  reflects electors without main residence in the commune. Excluding these removes valid observations.
* The INSEE geography is **later than April 2022**, so 2022 is also exposed to (few) mergers.
* A chef-lieu whose commune nouvelle absorbed only small communes can keep a ratio in [0.4, 1.2] while the INSEE unit
  is larger: the ratio rule then keeps a mismatched unit.

Samples compared: **strict** = `main_sample`; **broad** = metropolitan, matched, valid density, CENP available, and no
positive evidence that the INSEE code now designates another territory (INSEE name ≠ election name, or ratio < 0.4 in
a département where a commune disappeared that year); **all matched** (sensitivity only) = no geography rule. The name
comparison is heuristic; an exact treatment would use INSEE's *table de passage*.

In [ ]:
m22 = audit[(audit["year"] == 2022) & audit["metropolitan"]]
print("2022 metropolitan election communes without INSEE code:", int((~m22["insee_match"]).sum()),
      "→ if > 0, the INSEE commune geography is later than the April 2022 election")

cand = audit[audit["metropolitan"] & audit["valid_density"] & audit["CENP"].notna()].copy()
cand["ratio_class"] = np.select([cand["registered_to_population"] < 0.4, cand["registered_to_population"] > 1.2],
                                ["< 0.4", "> 1.2"], "0.4–1.2")
print("Name mismatch × registered/population class (metropolitan, matched, valid density):")
display(pd.crosstab([cand["year"], cand["name_mismatch"]], cand["ratio_class"], margins=True))

audit["evidence_other_unit"] = audit["insee_match"] & (audit["name_mismatch"]
                                                       | (audit["ratio_low"] & audit["dep_has_unmatched"]))
audit["all_matched_sample"] = audit["metropolitan"] & audit["valid_density"] & audit["CENP"].notna()
audit["broad_sample"] = audit["all_matched_sample"] & ~audit["evidence_other_unit"]
audit["sample_membership"] = np.select(
    [audit["main_sample"] & audit["broad_sample"], audit["main_sample"], audit["broad_sample"]],
    ["strict and broad", "strict only", "broad only"], "neither")
display(audit[audit["all_matched_sample"]].groupby(["year", "sample_membership"]).agg(
    n=("commune_code", "size"), median_density=("density", "median"), median_expressed=("expressed", "median"),
    mean_CENP=("CENP", "mean"), mean_ratio=("registered_to_population", "mean")).round(3))

In [ ]:
SHOW = ["year", "department_code", "commune_code", "commune_name", "insee_name", "registered", "population",
        "registered_to_population", "density", "main_sample", "broad_sample"]
with pd.option_context("display.max_rows", 200):
    print("Name mismatches (check for spelling-only differences), largest electorates first:")
    display(audit[audit["name_mismatch"]].sort_values(["year", "registered"], ascending=[True, False])[SHOW].head(150))
    print("In strict but with a name mismatch:")
    display(audit[audit["main_sample"] & audit["name_mismatch"]].sort_values("registered", ascending=False)[SHOW].head(50))
    print("Excluded from strict only because registered/population > 1.2 (kept in broad):")
    display(audit[audit["exclusion_reason"] == "boundary flag: registered/population > 1.2"]
            .sort_values("registered", ascending=False)[SHOW].head(30))
    print("Excluded from strict because ratio < 0.4 but kept in broad (no merger in the département that year):")
    display(audit[audit["ratio_low"] & audit["broad_sample"]].sort_values("registered", ascending=False)[SHOW].head(30))

In [ ]:
strict = audit[audit["main_sample"]].copy()
broad = audit[audit["broad_sample"]].copy()
all_matched = audit[audit["all_matched_sample"]].copy()
SAMPLES = {"strict": strict, "broad": broad}

boundary_comparison = stats_by_threshold({**SAMPLES, "all matched (no geography rule)": all_matched})
print("CENP ~ log_density under the three geography rules:")
display(boundary_comparison[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se_HC1", "ols_p_HC1", "ols_R2"]].round(5))

## 4. Population measure for the 2002 density

| variable | definition | role |
|---|---|---|
| `population` (2002 rows) | INSEE `D99_POP`, 1999 census (*sans doubles comptes*) | registered/population boundary flag |
| `population_2002_interp` | `D99_POP × (P06_POP / D99_POP)^(3/7)`, ≈ spring 2002 | **2002 density** |
| `population` (2022 rows) | INSEE `P22_POP` | **2022 density** |

Earlier versions used the 1999 census count for the 2002 density. Since the surface is identical, the two log
densities differ only by $(3/7)\,\ln(\text{P06\_POP}/\text{D99\_POP})$. The cells check the construction and show how
much the correction moves densities and the 2002 results.

In [ ]:
d02 = audit[(audit["year"] == 2002) & audit["insee_match"]]
report("2002 `population` equals INSEE D99_POP (1999 census; kept, not used for density)",
       d02["population"].eq(d02["commune_code"].map(insee.set_index("commune_code")["D99_POP"])).all())
v02 = d02[d02["population_2002_interp"] > 0]
report("2002 `density` = population_2002_interp / surface_km2",
       np.allclose(v02["density"].to_numpy(float), (v02["population_2002_interp"] / v02["surface_km2"]).to_numpy(float)))


def with_census_1999_density(df):
    # same rows, 2002 density recomputed with the 1999 census count (previous version); 2022 rows unchanged
    out = df.copy()
    is02 = out["year"] == 2002
    out.loc[is02, "density"] = out.loc[is02, "population"] / out.loc[is02, "surface_km2"]
    with np.errstate(divide="ignore", invalid="ignore"):
        out["log_density"] = np.log(out["density"])
    return out[np.isfinite(out["log_density"])]


pop_cmp = audit[(audit["year"] == 2002) & audit["all_matched_sample"]].copy()
pop_cmp["density_1999"] = pop_cmp["population"] / pop_cmp["surface_km2"]
with np.errstate(divide="ignore", invalid="ignore"):
    pop_cmp["log_density_1999"] = np.log(pop_cmp["density_1999"])
finite = np.isfinite(pop_cmp["log_density_1999"])
pc = pop_cmp[finite]
pc = pc.assign(log_diff=pc["log_density"] - pc["log_density_1999"],
               decile_1999=pd.qcut(pc["log_density_1999"].rank(method="first"), 10, labels=False),
               decile_interp=pd.qcut(pc["log_density"].rank(method="first"), 10, labels=False))
print(f"2002 matched communes: {len(pop_cmp)}; 1999-census density not finite: {(~finite).sum()}")
display(pd.Series({
    "Pearson(log density interp, log density 1999)": stats.pearsonr(pc["log_density"], pc["log_density_1999"]).statistic,
    "Spearman(density interp, density 1999)": stats.spearmanr(pc["density"], pc["density_1999"]).statistic,
    "median |log difference|": pc["log_diff"].abs().median(),
    "share with |log difference| > 0.10 (≈10%)": (pc["log_diff"].abs() > 0.10).mean(),
    "share in the same density decile": (pc["decile_1999"] == pc["decile_interp"]).mean(),
}, name="value").to_frame())
display(pc["log_diff"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame("log(interp / 1999 density)"))
print("Largest differences:")
display(pc.reindex(pc["log_diff"].abs().sort_values(ascending=False).index)
          [["commune_code", "commune_name", "population", "population_2002_interp", "surface_km2", "density_1999",
            "density", "log_diff"]].head(20))

pop_results = stats_by_threshold({"strict, interpolated (current)": strict,
                                  "strict, 1999 census (previous)": with_census_1999_density(strict),
                                  "broad, interpolated (current)": broad,
                                  "broad, 1999 census (previous)": with_census_1999_density(broad)})
print("2002 CENP ~ log_density with each population measure:")
display(pop_results.xs(2002, level="year")[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se_HC1", "ols_p_HC1", "ols_R2"]].round(5))

POP_2002_FOR_DENSITY = merged.loc[(merged["year"] == 2002) & merged["insee_match"], "density_population_variable"].unique().tolist()
assert POP_2002_FOR_DENSITY == ["population_2002_interp"], POP_2002_FOR_DENSITY
POP_2002_FOR_DENSITY = POP_2002_FOR_DENSITY[0]

## 5. Density validation

Positive population, surface and density, finite log density, no duplicated commune-year, plausible extremes, and the
largest cities where expected. Paris, Lyon and Marseille must each appear once per year as a whole commune, and the
whole-commune figures must equal the sum of the arrondissements (INSEE populations and election votes).

In [ ]:
for name, df in {**SAMPLES, "all matched": all_matched}.items():
    for year in YEARS:
        d = df[df["year"] == year]
        report(f"{name} {year}: density population, surface, density > 0 and log density finite",
               (d["density_population"] > 0).all() and (d["surface_km2"] > 0).all() and (d["density"] > 0).all()
               and np.isfinite(d["log_density"]).all())
        report(f"{name} {year}: no duplicated commune-year", not d.duplicated(["year", "commune_code"]).any())

print("Density (inhabitants / km²) by sample and year:")
display(pd.concat({name: df.groupby("year")["density"].describe(percentiles=[0.01, 0.5, 0.99])
                   for name, df in {**SAMPLES, "all matched": all_matched}.items()}).round(2))

EXTREME_COLS = ["commune_code", "commune_name", "insee_name", "population", "population_2002_interp", "surface_km2",
                "density", "expressed", "CENP", "main_sample", "broad_sample"]
for year in YEARS:
    d = all_matched[all_matched["year"] == year]
    print(f"\n{year}: 20 least dense (all matched communes)")
    display(d.nsmallest(20, "density")[EXTREME_COLS])
    print(f"{year}: 20 most dense")
    display(d.nlargest(20, "density")[EXTREME_COLS])

In [ ]:
BIG_CITIES = {"75056": "Paris", "13055": "Marseille", "69123": "Lyon", "31555": "Toulouse", "06088": "Nice",
              "44109": "Nantes", "67482": "Strasbourg", "34172": "Montpellier", "33063": "Bordeaux", "59350": "Lille",
              "69266": "Villeurbanne", "92044": "Levallois-Perret", "92051": "Neuilly-sur-Seine", "94080": "Vincennes"}
city_rows = []
for year in YEARS:
    pct = all_matched[all_matched["year"] == year].set_index("commune_code")["density"].rank(pct=True)
    for code, city in BIG_CITIES.items():
        row = audit[(audit["year"] == year) & (audit["commune_code"] == code)]
        city_rows.append({"year": year, "commune_code": code, "city": city, "rows": len(row),
                          "density": row["density"].iloc[0] if len(row) else np.nan,
                          "density_percentile": 100 * pct.get(code, np.nan),
                          "registered_to_population": row["registered_to_population"].iloc[0] if len(row) else np.nan,
                          "strict": bool(row["main_sample"].iloc[0]) if len(row) else False,
                          "broad": bool(row["broad_sample"].iloc[0]) if len(row) else False})
cities = pd.DataFrame(city_rows)
display(cities.round(2))
report("each major city appears exactly once per year", cities["rows"].eq(1).all(), cities[cities["rows"] != 1])
report("major cities in the top 5% of density", cities["density_percentile"].ge(95).all(),
       cities[~cities["density_percentile"].ge(95)])

report("no arrondissement code in the election data",
       not results["commune_code"].isin(ARRONDISSEMENTS).any() and not audit["commune_code"].isin(ARRONDISSEMENTS).any())
report("no arrondissement page among the parsed commune pages",
       not results["source_url"].map(com.url_stem).str.contains(r"AR\d{2}$", case=False).any())
for year in YEARS:
    n = audit[(audit["year"] == year) & audit["commune_code"].isin(PLM)].groupby("commune_code").size()
    report(f"{year}: Paris, Lyon, Marseille each once", n.reindex(list(PLM)).eq(1).all(), n)

arr = insee_raw[insee_raw["CODGEO"].isin(ARRONDISSEMENTS)].copy()
arr["city"] = arr["CODGEO"].map(lambda c: "75056" if c.startswith("751") else "69123" if c.startswith("6938") else "13055")
plm_insee = (arr.groupby("city")[["D99_POP", "P06_POP", "P22_POP", "SUPERF"]].sum().add_suffix("_sum_arr")
             .join(insee_raw.set_index("CODGEO")[["D99_POP", "P06_POP", "P22_POP", "SUPERF"]].add_suffix("_whole")))
display(plm_insee)
for col in ["D99_POP", "P06_POP", "P22_POP", "SUPERF"]:
    report(f"INSEE {col}: whole commune = sum of arrondissements (±1%)",
           np.allclose(plm_insee[f"{col}_whole"], plm_insee[f"{col}_sum_arr"], rtol=0.01))
for year in YEARS:
    cmp = com.plm_check(results, year)  # arrondissement pages from the HTML cache
    report(f"{year}: election whole-commune votes = sum of arrondissements",
           cmp is not None and cmp["difference"].fillna(1).eq(0).all())

## 6. Small electorates and extreme CENP values

With few votes, shares are noisy, and noise mechanically **raises** concentration: if `n` votes are drawn from true
shares $p$, $E[\sum \hat p_j^2] = \sum p_j^2 + (1-\sum p_j^2)/n$. `CENP_noise_benchmark` is the CENP implied by that
formula for a commune voting exactly like the country, with the bin's median electorate. Small electorates are also the
least dense communes, so this can create a **negative** density–CENP slope unrelated to coordination: the last table
reports the density slope **within** each size bin.

In [ ]:
SIZE_BINS = [0, 50, 100, 250, 500, 1000, 5000, np.inf]
SIZE_LABELS = ["< 50", "50–99", "100–249", "250–499", "500–999", "1,000–4,999", "5,000+"]


def size_bin(expressed):
    return pd.cut(expressed, SIZE_BINS, right=False, labels=SIZE_LABELS)


national_shares = {year: (lambda v: v / v.sum())(results[results["year"] == year].groupby("candidate_clean")["votes"].sum())
                   for year in YEARS}


def cenp_noise_benchmark(n, year):
    hhi = (national_shares[year] ** 2).sum()
    K = EXPECTED_K[year]
    return (K - 1 / (hhi + (1 - hhi) / n)) / (K - 1)


size_rows = []
for sname, df in SAMPLES.items():
    for year in YEARS:
        d = df[df["year"] == year].assign(size_bin=lambda x: size_bin(x["expressed"]))
        lo, hi = d.loc[d["expressed"] >= 1000, "CENP"].quantile([0.05, 0.95])
        g = d.groupby("size_bin", observed=False)
        t = g.agg(n_communes=("commune_code", "size"), CENP_mean=("CENP", "mean"), CENP_median=("CENP", "median"),
                  CENP_sd=("CENP", "std"), density_mean=("density", "mean"), density_median=("density", "median"),
                  expressed_median=("expressed", "median"))
        t["share_CENP_outside_p5_p95_of_1000plus"] = g["CENP"].apply(lambda s: ((s < lo) | (s > hi)).mean())
        t["CENP_noise_benchmark"] = [cenp_noise_benchmark(n, year) if n > 0 else np.nan for n in t["expressed_median"]]
        size_rows.append(t.reset_index().assign(sample=sname, year=year))
size_table = pd.concat(size_rows).set_index(["sample", "year", "size_bin"])
print("National CENP (n → ∞):", {year: round(cenp_noise_benchmark(np.inf, year), 4) for year in YEARS})
display(size_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for i, year in enumerate(YEARS):
    t = size_table.loc[("strict", year)]
    x = np.arange(len(t))
    axes[0].plot(x, t["CENP_mean"], "o-", color=f"C{i}", lw=2, label=f"{year}: mean CENP")
    axes[0].plot(x, t["CENP_noise_benchmark"], "--", color=f"C{i}", lw=1.2, label=f"{year}: sampling-noise benchmark")
    axes[1].plot(x, t["CENP_sd"], "o-", color=f"C{i}", lw=2, label=str(year))
for ax, title, ylabel in zip(axes, ["Mean CENP by electorate size", "Dispersion of CENP by electorate size"],
                             ["Mean CENP", "Standard deviation of CENP"]):
    ax.set_xticks(range(len(SIZE_LABELS)))
    ax.set_xticklabels(SIZE_LABELS, rotation=30)
    ax.set_xlabel("Expressed votes")
    ax.set_ylabel(ylabel)
    ax.set_title(title + " (strict sample)", fontsize=10)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
show_plot(fig)

In [ ]:
# Density slope within each electorate-size bin (removes the size–density confound across bins)
within_rows = []
for sname, df in SAMPLES.items():
    for year in YEARS:
        d = df[df["year"] == year].assign(size_bin=lambda x: size_bin(x["expressed"]))
        for b, g in d.groupby("size_bin", observed=True):
            if len(g) >= 30:
                within_rows.append({"sample": sname, "year": year, "size_bin": b, **association_stats(g, "CENP")})
within_size = pd.DataFrame(within_rows).set_index(["sample", "year", "size_bin"])
display(within_size[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se_HC1", "ols_p_HC1"]].round(5))

rows = []
for sname, df in SAMPLES.items():
    for year in YEARS:
        d = df[df["year"] == year].assign(size_bin=lambda x: size_bin(x["expressed"]).astype(str))
        fit = smf.ols("CENP ~ log_density + C(size_bin)", data=d).fit(cov_type="HC1")
        rows.append({"sample": sname, "year": year, "n": int(fit.nobs), "coef_log_density": fit.params["log_density"],
                     "se_HC1": fit.bse["log_density"], "p_HC1": fit.pvalues["log_density"]})
print("CENP ~ log_density + C(size bin):")
display(pd.DataFrame(rows).set_index(["sample", "year"]).round(5))

## 7. Density–coordination relationship by electorate threshold

All valid communes, then `expressed ≥ 100`, `≥ 500`, `≥ 1000`; all thresholds shown, none preferred.

## 8. Vote-weighted robustness (WLS, weights = expressed)

Robustness only: the unweighted regression describes the *typical commune*, the weighted one the *typical voter*.

In [ ]:
threshold_results = stats_by_threshold(SAMPLES, "CENP", weighted=True)
threshold_summary = threshold_results[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se", "ols_p",
                                       "ols_se_HC1", "ols_p_HC1", "ols_R2"]]
display(threshold_summary.round(5))
threshold_summary.to_csv(AUDIT_DIR / "audit_cenp_density_by_threshold.csv", encoding="utf-8")

In [ ]:
wls_comparison = threshold_results[["n", "ols_coef", "ols_se_HC1", "ols_p_HC1", "wls_coef", "wls_se_HC1", "wls_p_HC1", "wls_R2"]].copy()
wls_comparison["wls_minus_ols"] = wls_comparison["wls_coef"] - wls_comparison["ols_coef"]
display(wls_comparison.round(5))

## 9. Density deciles

Deciles of `log_density` (ranks, ties split) within each year and sample restriction. A roughly monotonic profile
means the slope is not driven by a few very dense communes. Compare the **shape** of the two years' profiles.

In [ ]:
def decile_table(df):
    d = df.copy()
    d["density_decile"] = d.groupby("year")["log_density"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 10, labels=False) + 1)
    return d.groupby(["year", "density_decile"]).agg(
        n_communes=("commune_code", "size"), density_median=("density", "median"),
        CENP_mean=("CENP", "mean"), CENP_median=("CENP", "median"),
        cliff_magnitude_mean=("cliff_magnitude", "mean"), cliff_ratio_mean=("cliff_ratio", "mean"),
        expressed_median=("expressed", "median"))


DECILE_RESTRICTIONS = {"all valid": 0, "expressed ≥ 500": 500}
decile_tables = {(sname, label): decile_table(df[df["expressed"] >= t])
                 for sname, df in SAMPLES.items() for label, t in DECILE_RESTRICTIONS.items()}
for key, t in decile_tables.items():
    print(f"\n{key[0]} sample, {key[1]}:")
    display(t.round(4))

monotonic = pd.DataFrame({key: {year: stats.spearmanr(t.loc[year].index, t.loc[year]["CENP_mean"]).statistic for year in YEARS}
                          for key, t in decile_tables.items()}).T
monotonic.index.names = ["sample", "restriction"]
print("Spearman(decile, mean CENP):")
display(monotonic.round(3))

In [ ]:
fig, axes = plt.subplots(len(SAMPLES), len(DECILE_RESTRICTIONS), figsize=(12, 4 * len(SAMPLES)), squeeze=False)
for i, sname in enumerate(SAMPLES):
    for j, label in enumerate(DECILE_RESTRICTIONS):
        ax, t = axes[i, j], decile_tables[(sname, label)]
        for k, year in enumerate(YEARS):
            ax.plot(t.loc[year].index, t.loc[year]["CENP_mean"], "o-", color=f"C{k}", lw=2, label=str(year))
        ax.set_xticks(range(1, 11))
        ax.set_xlabel("Density decile within the year (1 = least dense)")
        ax.set_ylabel("Mean CENP")
        ax.set_title(f"Mean CENP by density decile — {sname}, {label}", fontsize=10)
        ax.grid(axis="y", alpha=0.3)
        ax.legend(fontsize=8)
show_plot(fig)

## 10. Within départements: département fixed effects

$$\text{CENP}_i = \beta\,\log\text{density}_i + \alpha_{d(i)} + \varepsilon_i$$

The département dummies absorb everything shared by communes of the same département (regional traditions,
candidates' home bases, the département's average urbanisation): $\beta$ answers *within the same département, are
denser communes more coordinated?* Communes with `expressed ≥ 500` and `≥ 1000`; HC1 and département-clustered SE.

In [ ]:
FE_THRESHOLDS = [500, 1000]
fe_table = pd.DataFrame([{"sample": sname, "year": year, "min_expressed": t,
                          **fe_stats(df[(df["year"] == year) & (df["expressed"] >= t)])}
                         for sname, df in SAMPLES.items() for year in YEARS for t in FE_THRESHOLDS]
                        ).set_index(["sample", "year", "min_expressed"])
display(fe_table.round(5))

## 11. Pooled comparison of the 2002 and 2022 slopes

$\beta_3$ (`log_density:C(year)[T.2022]`) = 2022 slope − 2002 slope, on stacked cross-sections; HC1 and
département-clustered SE.

In [ ]:
POOLED_THRESHOLDS = [0, 500, 1000]
TERM = "log_density:C(year)[T.2022]"
pooled_rows = []
for sname, df in SAMPLES.items():
    for t in POOLED_THRESHOLDS:
        d = df.loc[df["expressed"] >= t, ["CENP", "log_density", "year", "department_code"]].dropna()
        hc1 = smf.ols("CENP ~ log_density * C(year)", data=d).fit(cov_type="HC1")
        cl = smf.ols("CENP ~ log_density * C(year)", data=d).fit(cov_type="cluster", cov_kwds={"groups": d["department_code"]})
        pooled_rows.append({"sample": sname, "min_expressed": t, "n": int(hc1.nobs),
                            "slope_2002": hc1.params["log_density"],
                            "slope_2022": hc1.params["log_density"] + hc1.params[TERM],
                            "interaction_2022": hc1.params[TERM], "se_HC1": hc1.bse[TERM], "p_HC1": hc1.pvalues[TERM],
                            "se_cluster_dep": cl.bse[TERM], "p_cluster_dep": cl.pvalues[TERM], "R2": hc1.rsquared})
pooled_table = pd.DataFrame(pooled_rows).set_index(["sample", "min_expressed"])
display(pooled_table.round(5))
print("Density slope by year (separate cross-sections, OLS, HC1):")
display(threshold_results[["n", "ols_coef", "ols_se_HC1", "ols_p_HC1"]].unstack("year").round(5))

## 12. Secondary measures: cliff magnitude, cliff ratio, cliff location

`cliff_location` is a position, so it is described by distributions, not regressions.

In [ ]:
CLIFF_VARS = ["cliff_magnitude", "cliff_ratio"]
cliff_threshold = pd.concat({y: stats_by_threshold(SAMPLES, y) for y in CLIFF_VARS}, names=["outcome"])
display(cliff_threshold[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se_HC1", "ols_p_HC1", "ols_R2"]].round(5))

cliff_fe = pd.DataFrame([{"outcome": y, "sample": sname, "year": year, "min_expressed": t,
                          **fe_stats(df[(df["year"] == year) & (df["expressed"] >= t)], y)}
                         for y in CLIFF_VARS for sname, df in SAMPLES.items() for year in YEARS for t in FE_THRESHOLDS]
                        ).set_index(["outcome", "sample", "year", "min_expressed"])
display(cliff_fe[["n", "coef", "se_HC1", "p_HC1", "coef_without_FE"]].round(5))

In [ ]:
for label, t in DECILE_RESTRICTIONS.items():
    d = strict[strict["expressed"] >= t]
    print(f"\ncliff_location — share of communes by year (strict, {label}):")
    display(pd.crosstab(d["cliff_location"], d["year"], normalize="columns").round(3))
    d = d.assign(density_quintile=d.groupby("year")["log_density"].transform(
        lambda s: pd.qcut(s.rank(method="first"), 5, labels=False) + 1))
    for year in YEARS:
        sub = d[d["year"] == year]
        print(f"{year}: cliff_location by density quintile (row shares; 1 = least dense), strict, {label}")
        display(pd.crosstab(sub["density_quintile"], sub["cliff_location"], normalize="index").round(3))

## Audit summary

Computed from the cells above — nothing is hard-coded. With tens of thousands of communes, tiny slopes are
"significant", so the **magnitude** columns of the scorecard (CENP change from the 25th to the 75th percentile of log
density, in CENP units and in standard deviations) matter as much as the p-values.

In [ ]:
print("1. Scrape failures")
display(scrape_summary[["expected_communes", "scraped_communes", "failed_communes", "pct_recovered",
                        "territories_without_commune_pages", "expected_from"]])
print(f"   Communes scraped but with 0 expressed votes: {zero_expressed.groupby('year').size().to_dict()}")

print("\n2. Sample sizes (metropolitan)")
sizes = sample_counts[["metropolitan_election_obs", "valid_density", "main_sample"]].rename(columns={"main_sample": "strict"})
sizes["broad"] = broad.groupby("year").size()
sizes["all_matched"] = all_matched.groupby("year").size()
for col in ["strict", "broad"]:
    sizes[f"pct_{col}"] = (100 * sizes[col] / sizes["metropolitan_election_obs"]).round(2)
display(sizes)

print("\n3. Exclusions from the strict sample by primary reason (metropolitan observations)")
display(pd.concat({"n": exclusion_counts.drop(index=["overseas", RETAINED]),
                   "% of metropolitan": exclusion_pct_metro.drop(index=RETAINED)}, axis=1))

print("\n4. Population used for the 2002 density:", POP_2002_FOR_DENSITY)
display(pop_results.xs(2002, level="year")[["n", "ols_coef", "ols_se_HC1", "pearson_r"]].unstack("min_expressed").round(4))

print("\n5. CENP–density correlations and OLS slopes by threshold and year")
display(threshold_results[["n", "pearson_r", "spearman_rho", "ols_coef", "ols_se_HC1", "ols_p_HC1"]].unstack("year").round(4))

print("\n6. Within-département (FE) slopes")
display(fe_table[["n", "coef", "se_HC1", "p_HC1", "p_cluster_dep", "coef_without_FE"]].unstack("year").round(4))

print("\n7. Pooled year × density interaction")
display(pooled_table[["n", "slope_2002", "slope_2022", "interaction_2022", "se_cluster_dep", "p_cluster_dep"]].round(4))

In [ ]:
# Robustness scorecard: every pre-specified specification, same yes/no questions
iqr = {year: strict.loc[strict["year"] == year, "log_density"].quantile(0.75)
             - strict.loc[strict["year"] == year, "log_density"].quantile(0.25) for year in YEARS}
sd_cenp = {year: strict.loc[strict["year"] == year, "CENP"].std() for year in YEARS}

score = []
for (sname, t), g in threshold_results.reset_index().groupby(["sample", "min_expressed"]):
    r = g.set_index("year")
    for spec, c, p in [("OLS", "ols_coef", "ols_p_HC1"), ("WLS (robustness)", "wls_coef", "wls_p_HC1")]:
        score.append({"specification": spec, "sample": sname, "min_expressed": t,
                      "slope_2002": r.at[2002, c], "p_2002": r.at[2002, p], "slope_2022": r.at[2022, c], "p_2022": r.at[2022, p]})
for (sname, t), g in fe_table.reset_index().groupby(["sample", "min_expressed"]):
    r = g.set_index("year")
    score.append({"specification": "département FE", "sample": sname, "min_expressed": t,
                  "slope_2002": r.at[2002, "coef"], "p_2002": r.at[2002, "p_HC1"],
                  "slope_2022": r.at[2022, "coef"], "p_2022": r.at[2022, "p_HC1"]})
score = pd.DataFrame(score).set_index(["specification", "sample", "min_expressed"])
for year in YEARS:
    score[f"IQR_effect_{year}"] = score[f"slope_{year}"] * iqr[year]
    score[f"IQR_effect_{year}_in_SD"] = score[f"IQR_effect_{year}"] / sd_cenp[year]
score["2022 > 0 (p<0.05)"] = (score["slope_2022"] > 0) & (score["p_2022"] < ALPHA)
score["2002 > 0 (p<0.05)"] = (score["slope_2002"] > 0) & (score["p_2002"] < ALPHA)
score["2002 < 0 (p<0.05)"] = (score["slope_2002"] < 0) & (score["p_2002"] < ALPHA)
score["2022 slope > 2002 slope"] = score["slope_2022"] > score["slope_2002"]
display(score.round(4))

flags = ["2022 > 0 (p<0.05)", "2002 > 0 (p<0.05)", "2002 < 0 (p<0.05)", "2022 slope > 2002 slope"]
print("Number of specifications where each pattern holds (out of", len(score), "):")
display(score[flags].sum().to_frame("n_specifications").T)
display(score.groupby(level="specification")[flags].sum())

pooled_flags = pooled_table.assign(**{"interaction > 0 (p<0.05, clustered)":
                                      (pooled_table["interaction_2022"] > 0) & (pooled_table["p_cluster_dep"] < ALPHA)})
display(pooled_flags[["n", "interaction_2022", "p_cluster_dep", "interaction > 0 (p<0.05, clustered)"]].round(5))

score.to_csv(AUDIT_DIR / "audit_scorecard.csv", encoding="utf-8")
audit[["year", "department_code", "commune_code", "commune_name", "insee_name", "exclusion_reason", "name_mismatch",
       "ratio_low", "ratio_high", "dep_has_unmatched", "evidence_other_unit", "main_sample", "broad_sample",
       "all_matched_sample"]].to_csv(AUDIT_DIR / "audit_sample_flags.csv", index=False, encoding="utf-8")
print("Audit tables written to", AUDIT_DIR)